Evaluate the presence of factual claim-making in SemEval articles and the relationship between propaganda and claim-making.

In [19]:
import pandas as pd
import spacy
from scipy import stats
import gc
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import logging
import transformers
import warnings

In [20]:
#Import functions for claim extraction
BASE_DIR = Path("../").resolve()
utils_path = str(BASE_DIR / "create-model")
if utils_path not in sys.path:
    sys.path.append(utils_path)
from claim_utils import lightweight_claimify

In [21]:
#Silence transformers and fastcoref
transformers.logging.set_verbosity_error()
logging.getLogger("fastcoref").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

In [22]:
#Load cleaned SemEval datasets
DATA_PATH = BASE_DIR / "data" / "interim" / "semeval_task2_tc_merged.csv"
df_merged = pd.read_csv(DATA_PATH)

#Define the persistent save path
processed_data_path = BASE_DIR / "data" / "interim" / "propaganda_vs_fact_stats.csv"

In [23]:
#Load lightweight spaCy model
nlp = spacy.load("en_core_web_sm")

In [24]:
#Group gold spans and unique articles
gold_spans_dict = df_merged.groupby('article_id').apply(
    lambda x: list(zip(x['start_char'].astype(int), x['end_char'].astype(int), x['technique']))
).to_dict()

unique_articles = df_merged[['article_id', 'text_content']].drop_duplicates('article_id')

In [25]:
#Analysis logic
def has_overlap(sent_start, sent_end, spans):
    for span_start, span_end in spans:
        if span_start < sent_end and span_end > sent_start:
            return True
    return False

In [26]:
def get_propaganda_data(sent_start, sent_end, article_spans):
    found_techs = []
    for s_start, s_end, tech in article_spans:
        if s_start < sent_end and s_end > sent_start:
            found_techs.append(tech)
    return len(found_techs) > 0, list(set(found_techs))

In [27]:
if processed_data_path.exists():
    df_existing = pd.read_csv(processed_data_path)
    processed_ids = set(df_existing['article_id'].unique())
    print(f"Resuming: {len(processed_ids)} articles already completed.")
    gc.collect()
else:
    processed_ids = set()

Resuming: 158 articles already completed.


In [ ]:
remaining_articles = unique_articles[~unique_articles['article_id'].isin(processed_ids)]
print(f"""{len(remaining_articles)} articles still need to be processed""")

for i, (_, row) in enumerate(tqdm(remaining_articles.iterrows(), total=len(remaining_articles))):
    aid = row['article_id']
    text = str(row['text_content'])
    spans = gold_spans_dict.get(aid, [])

    doc = nlp(text)
    total_sents = 0
    art_counts = {"prop_only": 0, "fact_only": 0, "both": 0, "neither": 0}

    #List to store techniques specifically used in 'both' sentences
    techniques_in_both = []

    for sent in doc.sents:
        total_sents += 1

        is_prop, techs_found = get_propaganda_data(sent.start_char, sent.end_char, spans)

        extracted_claims = lightweight_claimify(sent.text)
        is_fact = len(extracted_claims) > 0

        if is_prop and is_fact:
            art_counts["both"] += 1
            techniques_in_both.extend(techs_found)
        elif is_prop:
            art_counts["prop_only"] += 1
        elif is_fact:
            art_counts["fact_only"] += 1
        else:
            art_counts["neither"] += 1

    if total_sents > 0:
        #Create a unique, comma-separated list of techniques used in 'both' instances
        both_tech_summary = ",".join(sorted(set(techniques_in_both)))

        new_entry = pd.DataFrame([{
            "article_id": aid,
            "total_sentences": total_sents,
            "count_prop_only": art_counts["prop_only"],
            "count_fact_only": art_counts["fact_only"],
            "count_both": art_counts["both"],
            "count_neither": art_counts["neither"],
            "techniques_in_both": both_tech_summary,
            "prop_density": ((art_counts["prop_only"] + art_counts["both"]) / total_sents) * 100,
            "fact_density": ((art_counts["fact_only"] + art_counts["both"]) / total_sents) * 100
        }])

        new_entry.to_csv(processed_data_path, mode='a', index=False, header=not processed_data_path.exists())

    #Memory Cleanup
    if (i + 1) % 2 == 0:
        del doc
        gc.collect()

199 articles still need to be processed


  0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

  1%|          | 1/199 [01:26<4:46:09, 86.71s/it]

In [ ]:
#Reload full data
df_stats = pd.read_csv(processed_data_path)

counts = {
    "Propaganda Only": df_stats['count_prop_only'].sum(),
    "Fact Only": df_stats['count_fact_only'].sum(),
    "Both": df_stats['count_both'].sum(),
    "Neither": df_stats['count_neither'].sum()
}

In [ ]:
#Percentage Breakdown (The Intersections)
plt.figure(figsize=(10, 7))
plt.pie(counts.values(), labels=counts.keys(), autopct='%1.1f%%', startangle=140, colors=sns.color_palette("viridis"))
plt.title("Distribution of Sentences: Propaganda vs. Factual Claims")
plt.show()

In [ ]:
#Calculate Pearson correlation and p-value
r, p_value = stats.pearsonr(df_stats['prop_density'], df_stats['fact_density'])

plt.figure(figsize=(10, 6))
sns.regplot(data=df_stats, x='prop_density', y='fact_density',
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})

#Dynamic title showing significance
significance = "Significant" if p_value < 0.05 else "Not Significant"
plt.title(f"Correlation: {r:.3f} (p = {p_value:.4f})\n{significance} Relationship: Prop Density vs. Fact Density")

plt.xlabel("% of Sentences containing Propaganda")
plt.ylabel("% of Sentences containing Factual Claims")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Correlation Coefficient (r): {r:.4f}")
print(f"P-value: {p_value:.4e}")

In [ ]:
#Get random sample of weaponized facts
sample_count = 0
for _, row in unique_articles.sample(20).iterrows():
    if sample_count >= 5: break

    doc = nlp(str(row['text_content']))
    spans = gold_spans_dict.get(row['article_id'], [])

    for sent in doc.sents:
        if has_overlap(sent.start_char, sent.end_char, spans):
            claims = lightweight_claimify(sent.text)
            if len(claims) > 0:
                print(f"FOUND BOTH: {sent.text[:150]}...")
                sample_count += 1
                if sample_count >= 5: break